# Talent_AI — Phase 1 Pipeline Walkthrough

Exercises the core pipeline end-to-end on a small sample of resumes:
parsing -> NLP extraction -> anonymization -> embeddings -> semantic ranking
(FAISS) vs. a TF-IDF keyword-matching baseline.

**Prerequisite:** run `python scripts/download_dataset.py` from the project root
first so `Dataset/Raw/` has resume PDFs. For the full-dataset Precision@K
evaluation (not just this small-sample demo), run `python scripts/build_index.py`
then `python scripts/evaluate.py`.

In [ ]:
import sys
from pathlib import Path

# Locate the project root (the directory containing 'src') regardless of where
# Jupyter's cwd happens to be.
p = Path.cwd()
while not (p / "src").exists() and p != p.parent:
    p = p.parent
PROJECT_ROOT = p
sys.path.insert(0, str(PROJECT_ROOT / "src"))
PROJECT_ROOT

In [ ]:
from talent_ai.config import DATASET_RAW_DIR
from talent_ai.parsing.resume_parser import extract_text
from talent_ai.extraction.nlp_extractor import extract_all
from talent_ai.extraction.anonymize import anonymize_text
from talent_ai.embeddings.embedder import embed_texts
from talent_ai.matching.ranker import SemanticRanker
from talent_ai.matching.baseline import TfidfRanker
from talent_ai.schemas import CandidateProfile, JobDescription

In [ ]:
pdf_paths = sorted(DATASET_RAW_DIR.rglob("*.pdf"))[:20]
print(f"Using {len(pdf_paths)} sample resumes (run scripts/download_dataset.py first if this is 0)")

## Parse -> extract -> anonymize

Same steps `scripts/build_index.py` runs over the full dataset, on a small sample
here so the notebook runs quickly.

In [ ]:
profiles = []
for pdf_path in pdf_paths:
    raw_text = extract_text(pdf_path)
    if not raw_text.strip():
        continue
    anonymized = anonymize_text(raw_text)
    extracted = extract_all(raw_text)
    profiles.append(
        CandidateProfile(
            candidate_id=pdf_path.stem,
            source_path=str(pdf_path.relative_to(DATASET_RAW_DIR)),
            category=pdf_path.parent.name,
            raw_text=raw_text,
            anonymized_text=anonymized,
            skills=extracted["skills"],
            education=extracted["education"],
            experience=extracted["experience"],
        )
    )
len(profiles)

In [ ]:
# Inspect one profile — note the anonymized_text has names/emails/phones redacted
profiles[0].category, profiles[0].skills, profiles[0].education[:3], profiles[0].experience[:3]

## Embed

In [ ]:
embeddings = embed_texts([p.anonymized_text for p in profiles])
for profile, emb in zip(profiles, embeddings):
    profile.embedding = emb.tolist()
embeddings.shape

## Rank: semantic (FAISS) vs. TF-IDF baseline

In [ ]:
jd_path = PROJECT_ROOT / "scripts" / "sample_jds" / "information_technology.txt"
job = JobDescription(title=jd_path.stem, raw_text=jd_path.read_text(encoding="utf-8"))

semantic = SemanticRanker()
semantic.fit(profiles)
tfidf = TfidfRanker()
tfidf.fit(profiles)

semantic_results = semantic.rank(job, top_k=5)
tfidf_results = tfidf.rank(job, top_k=5)

profiles_by_id = {p.candidate_id: p for p in profiles}
for label, results in [("Semantic", semantic_results), ("TF-IDF", tfidf_results)]:
    print(f"\n{label} top 5:")
    for r in results:
        cand = profiles_by_id[r.candidate_id]
        print(f"  {r.rank}. {r.candidate_id} ({cand.category}) score={r.score:.3f}")

## Next steps

This notebook demos the pipeline on a small sample. For the real signal — Precision@K
of semantic vs. keyword ranking across the *full* dataset against category-based
ground truth — run:

```bash
python scripts/build_index.py
python scripts/evaluate.py
```

See `README.md` for the phased roadmap (Phase 2: LLM insights, Phase 3: dashboard,
Phase 4: automation).